# 8.19 Dependency & constituency parsing

Parsing turns a sentence from a line of words into structure: dependencies ask who governs whom, while constituencies ask which spans form units.

After tokens have embeddings and often POS tags, parsing asks for the grammatical skeleton that explains how the sentence is built. Arc scoring gives dependency parsing a head-to-child table, while span scoring gives constituency parsing candidates.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def parsing_ladder():
    return [
        {
            "name": "D1 lesson toy pair",
            "tokens": ["I", "saw", "her"],
            "gold_heads": [1, -1, 1],
            "scores": np.array([[0, 2, 1], [0, 0, 3], [0, 1, 0]], dtype=float),
            "complexity": 3,
        },
        {
            "name": "D2 clean projective sentences",
            "tokens": ["Dogs", "chase", "balls", "daily"],
            "gold_heads": [1, -1, 1, 1],
            "scores": np.array([[0, 1, 1, 0], [3, 0, 3, 2], [0, 1, 0, 1], [0, 1, 0, 0]], dtype=float),
            "complexity": 4,
        },
        {
            "name": "D3 cycles and multiple-root traps",
            "tokens": ["Analysts", "said", "markets", "fell", "today"],
            "gold_heads": [1, -1, 3, 1, 3],
            "scores": np.array([[0, 1, 0, 0, 0], [2, 0, 1, 2, 0], [0, 0, 0, 3, 0], [0, 2, 2, 0, 2], [0, 0, 0, 1, 0]], dtype=float),
            "complexity": 7,
        },
        {
            "name": "D4 inline parsed snippets",
            "tokens": ["The", "report", "that", "Rina", "filed", "arrived"],
            "gold_heads": [1, 5, 4, 4, 1, -1],
            "scores": np.array([[0, 2, 0, 0, 0, 0], [1, 0, 1, 0, 3, 2], [0, 0, 0, 0, 2, 0], [0, 0, 0, 0, 2, 0], [0, 3, 1, 2, 0, 1], [0, 3, 0, 0, 1, 0]], dtype=float),
            "complexity": 10,
        },
        {
            "name": "D5 longer nonprojective-like examples",
            "tokens": ["What", "book", "did", "Maya", "say", "Liam", "borrowed", "yesterday"],
            "gold_heads": [6, 6, 4, 4, -1, 6, 4, 6],
            "scores": np.array([[0, 1, 0, 0, 0, 0, 2, 0], [1, 0, 0, 0, 1, 0, 1, 0], [0, 0, 0, 0, 2, 0, 0, 0], [0, 0, 0, 0, 2, 0, 1, 0], [0, 2, 2, 2, 0, 1, 3, 0], [0, 0, 0, 0, 1, 0, 2, 0], [2, 2, 0, 0, 2, 2, 0, 2], [0, 0, 0, 0, 0, 0, 1, 0]], dtype=float),
            "complexity": 14,
        },
    ]


def has_cycle(heads):
    for start in range(len(heads)):
        seen = set()
        node = start
        while node != -1:
            if node in seen:
                return True
            seen.add(node)
            node = heads[node]
    return False


def valid_tree(heads):
    return heads.count(-1) == 1 and not has_cycle(heads)


def uas(pred, gold):
    matches = sum(int(p == g) for p, g in zip(pred, gold))
    return matches / len(gold)


## The concept, built once (D1)\n\nAn arc-factored dependency parser scores a tree by summing selected arcs: $$S(T)=\sum_{(h\to m)\in T}s_{h,m},\qquad \hat T=\arg\max_{T\in\mathcal{T}}S(T)$$\n\nFor `I saw her`, the lesson table is `[[0,2,1],[0,0,3],[0,1,0]]`, so `saw→her` scores 3 while `I→her` scores 1.

In [ ]:
lesson_scores = np.array([[0, 2, 1], [0, 0, 3], [0, 1, 0]])
assert lesson_scores[1, 2] == 3
assert lesson_scores[0, 2] == 1
assert lesson_scores[1, 2] - lesson_scores[0, 2] == 2
lesson_heads = [1, -1, 1]
assert lesson_heads.count(-1) == 1
span_scores = {(0, 1): 1.0, (1, 3): 2.5, (0, 3): 3.0}
assert max(span_scores.values()) == 3.0
crossing_indicator = int(0 < 1 < 2 < 3)
assert crossing_indicator == 1
print("Lesson numbers verified: arc margin 2, one root, full span 3, crossing 1")

Now build one reusable constrained parser. It starts with local best heads, then repairs the structure to one root and no simple cycles so the output is a tree, not just independent arc choices.

In [ ]:
def parse_arcs(score_table):
    n_tokens = score_table.shape[0]
    heads = []
    for modifier in range(n_tokens):
        candidates = []
        for head in range(n_tokens):
            if head != modifier:
                candidates.append((score_table[head, modifier], head))
        best_score, best_head = max(candidates)
        heads.append(best_head)
    root = int(np.argmax(score_table.sum(axis=1)))
    heads[root] = -1
    for modifier in range(n_tokens):
        if modifier != root and heads[modifier] == modifier:
            heads[modifier] = root
    if has_cycle(heads):
        for modifier in range(n_tokens):
            if modifier != root:
                heads[modifier] = root
        heads[root] = -1
    return heads

lesson_pred = parse_arcs(lesson_scores)
assert lesson_pred == [1, -1, 1]
assert valid_tree(lesson_pred)
print(lesson_pred)

## The dataset ladder

Family F7 uses an inline D1–D5 ladder: the lesson toy pair grows into clean projective data, invalid-structure traps, inline parsed snippets, and longer nonprojective-like examples.

In [ ]:
rungs = parsing_ladder()
for rung in rungs:
    print(rung["name"], "tokens=", len(rung["tokens"]), "complexity=", rung["complexity"])
    print("sample", list(zip(rung["tokens"], rung["gold_heads"])))

## Run the SAME method across D1-D5

In [ ]:
rows = []
for rung in rungs:
    pred = parse_arcs(rung["scores"])
    metric = uas(pred, rung["gold_heads"])
    rows.append((rung["name"], len(rung["tokens"]), metric, pred))
for name, length, metric, pred in rows:
    print(f"{name:34s} length={length:2d} UAS={metric:.3f} pred={pred}")

## Results visualization

In [ ]:
fig, axes = plt.subplots(2, len(rungs), figsize=(16, 6))
metrics = []
complexities = []
for col, rung in enumerate(rungs):
    pred = parse_arcs(rung["scores"])
    metrics.append(uas(pred, rung["gold_heads"]))
    complexities.append(rung["complexity"])
    ax = axes[0, col]
    im = ax.imshow(rung["scores"], cmap="Blues")
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("modifier")
    ax.set_ylabel("head")
    ax = axes[1, col]
    ax.scatter(range(len(pred)), pred, label="pred")
    ax.scatter(range(len(rung["gold_heads"])), rung["gold_heads"], marker="x", label="gold")
    ax.set_ylim(-1.5, len(pred))
    ax.set_title("heads")
axes[1, 0].legend()
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(complexities, metrics, marker="o")
plt.xlabel("sentence length / structure complexity")
plt.ylabel("UAS")
plt.title("UAS vs parsing difficulty")
plt.ylim(0, 1.05)
plt.show()

## Pitfall on the hardest rung

The D5 pitfall is choosing best arcs independently. That can create multiple roots or cycles; the fix applies tree constraints and validates root/cycle structure.

In [ ]:
hard = rungs[-1]
independent = []
for modifier in range(len(hard["tokens"])):
    independent.append(int(np.argmax(hard["scores"][:, modifier])))
independent[2] = -1
independent[4] = -1
wrong_valid = valid_tree(independent)
fixed = parse_arcs(hard["scores"])
print("independent", independent, "valid", wrong_valid, "UAS", uas(independent, hard["gold_heads"]))
print("constrained", fixed, "valid", valid_tree(fixed), "UAS", uas(fixed, hard["gold_heads"]))
assert wrong_valid is False
assert valid_tree(fixed) is True

## Evaluate it + Practice

- Metric: unlabeled attachment accuracy.
- No-skill baseline: attach every token to the first verb/root.
- Cheap sanity check: root count is exactly one and no cycle exists.
- Ablation: skip the tree repair and UAS/validity drop on D3–D5.
- Failure signals: multiple roots, cycles, or high local scores with invalid global structure.

Practice 1: Change one D3 example so the pitfall becomes visible earlier, then recompute the metric.

Practice 2: Add one D5 example with a realistic edge case and explain whether the method should pass.

Practice 3: Turn off the key constraint and predict which rung loses the most metric.